# FlowCycle 改进版：摊销化噪声预测（Amortized Noise Prediction）

原版 FlowCycle 对每张图都要从随机噪声开始，梯度优化 100 轮（~10 分钟/张），且无法泛化到新图。  
改进：训练一个小型卷积网络 **NoisePredictor**，一次前向传播直接输出噪声。

**三阶段流程：**
1. **生成伪标签**：用原版 FlowCycle 逐图优化，保存优化好的噪声
2. **训练 NoisePredictor**：用 MSE 监督学习伪标签噪声（不跑 SD3，几秒完成）
3. **推理**：一次前向传播预测噪声 → 去噪得到编辑结果

**架构：** Conv 编码器 → 4×AdaGN ResBlock（文本条件注入）→ Conv 解码器  
**优势：** 质量与原版持平，推理速度提升数个量级

## Step 1: 检查 GPU & 安装依赖

In [ ]:
!nvidia-smi
import torch
print(f"\nGPU: {torch.cuda.get_device_name(0)}")
print(f"bfloat16 支持: {torch.cuda.is_bf16_supported()}")

In [ ]:
!pip install -q diffusers==0.30.1 transformers accelerate sentencepiece==0.2.0

## Step 2: 登录 HuggingFace

In [ ]:
from huggingface_hub import login
login()

## Step 3: 上传数据

上传数据，每组包含 `source.png` 和 `prompt.txt`（第一行源提示词，第二行目标提示词）。  
NoisePredictor 不需要 target 图像（无监督训练）。

In [ ]:
import os
from google.colab import files
from IPython.display import display
from PIL import Image

for t in data_dirs:
    os.makedirs(t, exist_ok=True)
    print(f"=== 请上传 {t}/source.png ===")
    for name, data in files.upload().items():
        with open(f"{t}/source.png", "wb") as f:
            f.write(data)
    print(f"=== 请上传 {t}/prompt.txt ===")
    for name, data in files.upload().items():
        with open(f"{t}/prompt.txt", "wb") as f:
            f.write(data)

# 预览上传的数据
import matplotlib.pyplot as plt
n = len(data_dirs)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
if n == 1:
    axes = [axes]
for i, t in enumerate(data_dirs):
    axes[i].imshow(Image.open(f"{t}/source.png"))
    with open(f"{t}/prompt.txt") as f:
        lines = f.read().strip().split("\n")
    axes[i].set_title(f"{t}\nsrc: ...{lines[0][-30:]}\ntar: ...{lines[1][-30:]}", fontsize=9)
    axes[i].axis("off")
plt.tight_layout()
plt.show()

## Step 4: 定义工具函数和 NoisePredictor 网络

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from diffusers import StableDiffusion3Pipeline, FlowMatchEulerDiscreteScheduler, AutoencoderKL
from diffusers.models.normalization import AdaGroupNorm
from PIL import Image
from typing import Union, Tuple, Optional
from torchvision import transforms as tvt
from torch.optim.lr_scheduler import SequentialLR, LinearLR, CosineAnnealingLR
import random
import numpy as np
import os
import matplotlib.pyplot as plt

# ======================== 工具函数 ========================

@torch.no_grad()
def img_to_latents(x, vae):
    x = 2. * x.to(device=vae.device, dtype=vae.dtype) - 1.
    posterior = vae.encode(x).latent_dist
    return posterior.mean * vae.config.scaling_factor

class Noises(nn.Module):
    """原版 FlowCycle 的可学习噪声（Phase 1 生成伪标签用）"""
    def __init__(self, dtype, img_size):
        super().__init__()
        self.noises = nn.Parameter(torch.randn(2, 16, img_size // 8, img_size // 8, dtype=dtype))
    def forward(self, i):
        return self.noises[i].unsqueeze(0)

def load_image(imgname, target_size=None):
    pil_img = Image.open(imgname).convert('RGB')
    if target_size is not None:
        if isinstance(target_size, int):
            target_size = (target_size, target_size)
        pil_img = pil_img.resize(target_size, Image.Resampling.LANCZOS)
    return tvt.ToTensor()(pil_img)[None, ...]

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

@torch.no_grad()
def get_text_embeddings(prompt_a, prompt_b, src_gs, tar_gs, pipe):
    pe, npe, ppe, nppe = pipe.encode_prompt(
        prompt=prompt_a, prompt_2=prompt_a, prompt_3=prompt_a,
        negative_prompt="", negative_prompt_2="", negative_prompt_3="",
        device=pipe.device, num_images_per_prompt=1,
        do_classifier_free_guidance=src_gs > 1.0)
    if src_gs > 1.0:
        ce_a = torch.cat([npe, pe], dim=0)
        pp_a = torch.cat([nppe, ppe], dim=0)
    else:
        ce_a, pp_a = pe, ppe

    pe, npe, ppe, nppe = pipe.encode_prompt(
        prompt=prompt_b, prompt_2=prompt_b, prompt_3=prompt_b,
        negative_prompt="", negative_prompt_2="", negative_prompt_3="",
        device=pipe.device, num_images_per_prompt=1,
        do_classifier_free_guidance=tar_gs > 1.0)
    if tar_gs > 1.0:
        ce_b = torch.cat([npe, pe], dim=0)
        pp_b = torch.cat([nppe, ppe], dim=0)
    else:
        ce_b, pp_b = pe, ppe
    return ce_a, pp_a, ce_b, pp_b

@torch.no_grad()
def get_pooled_embeddings(prompt, pipe):
    """获取单条 prompt 的 pooled embeddings（不做 CFG 拼接）"""
    _, _, ppe, _ = pipe.encode_prompt(
        prompt=prompt, prompt_2=prompt, prompt_3=prompt,
        negative_prompt="", negative_prompt_2="", negative_prompt_3="",
        device=pipe.device, num_images_per_prompt=1,
        do_classifier_free_guidance=False)
    return ppe

def decode_latent_to_pil(latent, vae):
    with torch.no_grad():
        img = vae.decode(latent / vae.config.scaling_factor, return_dict=False)[0]
        img = (img / 2 + 0.5).clamp(0, 1)
        img = img[0].permute(1, 2, 0).float().cpu().numpy()
        return Image.fromarray((img * 255).round().astype("uint8"))

def denoise_loop(latents, timesteps, pipe, cond_embeds, pooled_proj, gs, device):
    for _, t in enumerate(timesteps):
        t_idx = pipe.scheduler.timesteps.tolist().index(t.item())
        sigma_t = pipe.scheduler.sigmas[t_idx]
        sigma_next = pipe.scheduler.sigmas[t_idx + 1]
        dt = sigma_t - sigma_next
        inp = latents.repeat(2, 1, 1, 1) if gs > 1.0 else latents
        t_batch = t.expand(inp.shape[0]).to(device)
        with torch.inference_mode():
            pred = pipe.transformer(
                hidden_states=inp, timestep=t_batch,
                encoder_hidden_states=cond_embeds,
                pooled_projections=pooled_proj,
                return_dict=True).sample
        if gs > 1.0:
            u, c = pred.chunk(2)
            pred = u + gs * (c - u)
        latents = latents - dt * pred
    return latents

# ======================== NoisePredictor 网络 ========================

class ResBlock(nn.Module):
    """带 AdaGroupNorm 的残差块，用文本条件调制特征"""
    def __init__(self, ch, cond_dim):
        super().__init__()
        self.norm1 = AdaGroupNorm(cond_dim, ch, num_groups=8)
        self.conv1 = nn.Conv2d(ch, ch, 3, padding=1)
        self.norm2 = AdaGroupNorm(cond_dim, ch, num_groups=8)
        self.conv2 = nn.Conv2d(ch, ch, 3, padding=1)

    def forward(self, x, cond):
        h = self.norm1(x, cond)
        h = F.silu(h)
        h = self.conv1(h)
        h = self.norm2(h, cond)
        h = F.silu(h)
        h = self.conv2(h)
        return x + h

class NoisePredictor(nn.Module):
    """
    输入: img_latent (B,16,H,W) + pooled_src (B,D) + pooled_tar (B,D)
    输出: eps_src (B,16,H,W), eps_tar (B,16,H,W)
    """
    def __init__(self, in_ch=16, hidden_ch=64, cond_dim=128, pooled_dim=2048):
        super().__init__()
        self.text_proj = nn.Sequential(
            nn.Linear(pooled_dim, cond_dim),
            nn.SiLU(),
            nn.Linear(cond_dim, cond_dim),
        )
        self.encoder = nn.Conv2d(in_ch, hidden_ch, 3, padding=1)
        self.blocks = nn.ModuleList([ResBlock(hidden_ch, cond_dim) for _ in range(4)])
        self.decoder = nn.Conv2d(hidden_ch, in_ch * 2, 3, padding=1)

    def forward(self, img_latent, pooled_src, pooled_tar):
        cond = self.text_proj(pooled_tar - pooled_src)
        h = self.encoder(img_latent)
        for block in self.blocks:
            h = block(h, cond)
        out = self.decoder(h)
        eps_src, eps_tar = out.chunk(2, dim=1)
        return eps_src, eps_tar

print("所有函数和网络定义完成！")

## Step 5: 加载 SD3 模型 & 设置超参数

In [ ]:
# 超参数
model_id = "stabilityai/stable-diffusion-3-medium-diffusers"
seed = 2025
num_inference_steps = 50
n_max = 33
source_guidance_scale = 3.5
target_guidance_scale = 5.5
# Phase 1: 原版 FlowCycle 噪声优化参数
flowcycle_lr = 0.1
flowcycle_epochs = 100
alpha = 0.2
# Phase 2: NoisePredictor 训练参数
predictor_lr = 1e-3
predictor_epochs = 1000
img_size = 512
device = "cuda:0"
data_dirs = ["tuple1"]

# 自动选择精度
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f"使用精度: {dtype}")

set_seed(seed)

# 加载模型
print("加载 SD3 模型...")
pipe = StableDiffusion3Pipeline.from_pretrained(model_id, torch_dtype=dtype).to(device)
pipe.scheduler = FlowMatchEulerDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.scheduler.set_timesteps(num_inference_steps, device=device)
timesteps = pipe.scheduler.timesteps[num_inference_steps - n_max:]
sigma_max_t = pipe.scheduler.sigmas[pipe.scheduler.timesteps.tolist().index(timesteps[0])]
print("模型加载完成！")

## Phase 1: 用原版 FlowCycle 生成伪标签

对每组数据跑原版 FlowCycle 噪声优化（L_src + α·L_align），保存优化后的噪声作为伪标签。  
同时保存编辑结果 `edited_baseline.png` 用于后续对比。

In [ ]:
# ---- 预处理：编码所有数据 ----
all_data = []
for data_dir in data_dirs:
    with open(os.path.join(data_dir, "prompt.txt"), 'r') as f:
        lines = f.read().strip().split('\n')
    source_prompt, target_prompt = lines[0].strip(), lines[1].strip()

    input_img = load_image(os.path.join(data_dir, "source.png"), img_size).to(device=device, dtype=dtype)
    a = img_to_latents(input_img, pipe.vae)

    ce_a, pp_a, ce_b, pp_b = get_text_embeddings(
        source_prompt, target_prompt, source_guidance_scale, target_guidance_scale, pipe)
    pooled_src = get_pooled_embeddings(source_prompt, pipe)
    pooled_tar = get_pooled_embeddings(target_prompt, pipe)

    all_data.append({
        'name': data_dir,
        'img_latent': a,
        'pooled_src': pooled_src,
        'pooled_tar': pooled_tar,
        'ce_a': ce_a, 'pp_a': pp_a,
        'ce_b': ce_b, 'pp_b': pp_b,
    })
    print(f"已编码 {data_dir}: src={source_prompt[:40]}... tar={target_prompt[:40]}...")

# ---- Phase 1: 逐组跑原版 FlowCycle 优化 ----
warmup_epochs = int(flowcycle_epochs * 0.1)

for sample in all_data:
    data_dir = sample['name']
    print(f"\n{'='*50}")
    print(f"Phase 1: 优化 {data_dir} 的噪声 ({flowcycle_epochs} epochs)...")

    noises = Noises(dtype, img_size).to(device)
    optimizer = optim.Adam(noises.parameters(), lr=flowcycle_lr)
    warmup = LinearLR(optimizer, start_factor=1e-6, end_factor=1.0, total_iters=warmup_epochs)
    cosine = CosineAnnealingLR(optimizer, T_max=flowcycle_epochs - warmup_epochs, eta_min=1e-5)
    scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[warmup_epochs])

    a = sample['img_latent']
    for epoch in range(flowcycle_epochs):
        optimizer.zero_grad()
        eps_src = noises(0)
        eps_tar = noises(1)

        # Source → Target
        latents = a * (1 - sigma_max_t) + sigma_max_t * eps_src
        tmp_a = latents
        latents = denoise_loop(latents, timesteps, pipe, sample['ce_b'], sample['pp_b'],
                               target_guidance_scale, device)

        # Target → Source
        latents = latents * (1 - sigma_max_t) + sigma_max_t * eps_tar
        tmp_b = latents
        latents = denoise_loop(latents, timesteps, pipe, sample['ce_a'], sample['pp_a'],
                               source_guidance_scale, device)

        loss1 = F.mse_loss(a, latents, reduction='none').mean(dim=(1, 2, 3))
        loss2 = F.mse_loss(tmp_a, tmp_b, reduction='none').mean(dim=(1, 2, 3))
        loss = loss1 + alpha * loss2
        loss.backward()
        optimizer.step()
        scheduler.step()

        if epoch % 20 == 0:
            print(f"  epoch {epoch}/{flowcycle_epochs}, loss={loss.item():.6f}")

    # 保存伪标签噪声
    torch.save(noises.noises.data.cpu(), os.path.join(data_dir, "learned_noises.pt"))
    print(f"已保存 {data_dir}/learned_noises.pt")

    # 保存 baseline 编辑结果
    with torch.no_grad():
        eps_src = noises(0)
        latents = a * (1 - sigma_max_t) + sigma_max_t * eps_src
        latents = denoise_loop(latents, timesteps, pipe, sample['ce_b'], sample['pp_b'],
                               target_guidance_scale, device)
        baseline = decode_latent_to_pil(latents, pipe.vae)
        baseline.save(os.path.join(data_dir, "edited_baseline.png"))
    print(f"已保存 {data_dir}/edited_baseline.png")

print("\nPhase 1 完成！所有伪标签已生成。")

## Phase 2: 训练 NoisePredictor（MSE 监督）

用 Phase 1 保存的伪标签噪声做 MSE 监督，训练小网络直接预测噪声。  
不需要跑 SD3 transformer，纯小网络训练，几秒完成。

In [ ]:
# ---- 加载伪标签 ----
train_samples = []
for sample in all_data:
    gt_noises = torch.load(os.path.join(sample['name'], "learned_noises.pt"),
                           map_location=device).to(dtype)  # (2, 16, H, W)
    train_samples.append({
        'img_latent': sample['img_latent'],
        'pooled_src': sample['pooled_src'],
        'pooled_tar': sample['pooled_tar'],
        'gt_eps_src': gt_noises[0:1],  # (1, 16, H, W)
        'gt_eps_tar': gt_noises[1:2],  # (1, 16, H, W)
    })
    print(f"已加载 {sample['name']}/learned_noises.pt")

# ---- 初始化 NoisePredictor ----
pooled_dim = all_data[0]['pooled_src'].shape[-1]
predictor = NoisePredictor(in_ch=16, hidden_ch=64, cond_dim=128, pooled_dim=pooled_dim).to(device).to(dtype)
pred_opt = optim.Adam(predictor.parameters(), lr=predictor_lr)
pred_scheduler = CosineAnnealingLR(pred_opt, T_max=predictor_epochs, eta_min=1e-6)

param_count = sum(p.numel() for p in predictor.parameters())
print(f"NoisePredictor 参数量: {param_count:,}")

# ---- MSE 监督训练 ----
predictor.train()
losses_history = []
print(f"\n开始训练 NoisePredictor, {predictor_epochs} epochs (MSE 监督)\n")

for epoch in range(predictor_epochs):
    epoch_loss = 0.0
    for s in train_samples:
        pred_opt.zero_grad()
        pred_src, pred_tar = predictor(s['img_latent'], s['pooled_src'], s['pooled_tar'])
        loss = F.mse_loss(pred_src, s['gt_eps_src']) + F.mse_loss(pred_tar, s['gt_eps_tar'])
        loss.backward()
        pred_opt.step()
        epoch_loss += loss.item()
    pred_scheduler.step()

    avg_loss = epoch_loss / len(train_samples)
    losses_history.append(avg_loss)
    if epoch % 100 == 0:
        print(f"  epoch {epoch}/{predictor_epochs}, MSE={avg_loss:.6f}, lr={pred_opt.param_groups[0]['lr']:.2e}")

print("\nNoisePredictor 训练完成！")
torch.save(predictor.state_dict(), "noise_predictor.pt")

plt.figure(figsize=(8, 4))
plt.plot(losses_history)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("NoisePredictor Training (MSE vs pseudo-labels)")
plt.yscale("log")
plt.grid(True)
plt.show()

## Phase 3: 推理 + 对比可视化

用 NoisePredictor 预测噪声（一次前向传播）→ 去噪得到编辑结果。  
对比：Source | Random Noise | NoisePredictor | Baseline（原版 FlowCycle）

In [ ]:
predictor.eval()

for i, sample in enumerate(all_data):
    data_dir = sample['name']
    a = sample['img_latent']

    # NoisePredictor 推理
    with torch.no_grad():
        pred_eps_src, _ = predictor(a, sample['pooled_src'], sample['pooled_tar'])
        latents = a * (1 - sigma_max_t) + sigma_max_t * pred_eps_src
        latents = denoise_loop(latents, timesteps, pipe, sample['ce_b'], sample['pp_b'],
                               target_guidance_scale, device)
        pred_result = decode_latent_to_pil(latents, pipe.vae)
        pred_result.save(os.path.join(data_dir, "edited_predictor.png"))

    # 随机噪声对比
    with torch.no_grad():
        random_eps = torch.randn_like(a)
        latents_rand = a * (1 - sigma_max_t) + sigma_max_t * random_eps
        latents_rand = denoise_loop(latents_rand, timesteps, pipe, sample['ce_b'], sample['pp_b'],
                                    target_guidance_scale, device)
        rand_result = decode_latent_to_pil(latents_rand, pipe.vae)
        rand_result.save(os.path.join(data_dir, "edited_random.png"))

    # 可视化对比
    src_img = Image.open(os.path.join(data_dir, "source.png")).convert('RGB').resize((img_size, img_size))
    baseline_img = Image.open(os.path.join(data_dir, "edited_baseline.png")).convert('RGB')

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(src_img)
    axes[0].set_title(f"Source ({data_dir})")
    axes[0].axis("off")
    axes[1].imshow(rand_result)
    axes[1].set_title("Random Noise")
    axes[1].axis("off")
    axes[2].imshow(pred_result)
    axes[2].set_title("NoisePredictor (ours)")
    axes[2].axis("off")
    axes[3].imshow(baseline_img)
    axes[3].set_title("Baseline (FlowCycle)")
    axes[3].axis("off")
    plt.suptitle(f"Comparison - {data_dir}", fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(data_dir, "comparison.png"), dpi=150, bbox_inches='tight')
    plt.show()

print("Phase 3 完成！")

## 下载结果

In [ ]:
from google.colab import files
files.download("noise_predictor.pt")
for t in data_dirs:
    files.download(f"{t}/comparison.png")
    files.download(f"{t}/edited_predictor.png")
    files.download(f"{t}/edited_baseline.png")